In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold



In [2]:
df = pd.read_csv('heart.csv')

print(df.sample(6))

     age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  \
142   42    0   2       120   209    0        1      173      0      0.0   
22    42    1   0       140   226    0        1      178      0      0.0   
215   43    0   0       132   341    1        0      136      1      3.0   
217   63    1   0       130   330    1        0      132      1      1.8   
170   56    1   2       130   256    1        0      142      1      0.6   
256   58    1   0       128   259    0        0      130      1      3.0   

     slope  ca  thal  target  
142      1   0     2       1  
22       2   0     2       1  
215      1   0     3       0  
217      2   3     3       0  
170      1   1     1       0  
256      1   2     3       0  


In [3]:
missing_values = df.isnull().sum()
print("Missing values in each column:\n", missing_values)

numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
categorical_cols = df.select_dtypes(include=['object']).columns

for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mean(), inplace=True)

for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

print("\nRemaining missing values (should be zero):\n", df.isnull().sum().sum())

Missing values in each column:
 age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64

Remaining missing values (should be zero):
 0


In [4]:
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
categorical_cols = df.select_dtypes(include=['object']).columns

for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mean(), inplace=True)

for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)


In [5]:
scaler = StandardScaler()
scaled_features = scaler.fit_transform(df.drop('target', axis=1))

import numpy as np
X = pd.DataFrame(scaled_features, columns=df.drop('target', axis=1).columns)
y = df['target']

In [6]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [7]:

model_lr = LogisticRegression()
model_rf = RandomForestClassifier(random_state=42)
model_svc = SVC(probability=True)

model_lr.fit(X_train, y_train)
model_rf.fit(X_train, y_train)
model_svc.fit(X_train, y_train)

pred_lr = model_lr.predict(X_test)
pred_rf = model_rf.predict(X_test)
pred_svc = model_svc.predict(X_test)

acc_lr = accuracy_score(y_test, pred_lr)
acc_rf = accuracy_score(y_test, pred_rf)
acc_svc = accuracy_score(y_test, pred_svc)

f1_lr = f1_score(y_test, pred_lr)
f1_rf = f1_score(y_test, pred_rf)
f1_svc = f1_score(y_test, pred_svc)

results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'SVM'],
    'Accuracy': [acc_lr, acc_rf, acc_svc],
    'F1 Score': [f1_lr, f1_rf, f1_svc]
})

print(results)


                 Model  Accuracy  F1 Score
0  Logistic Regression  0.803279  0.833333
1        Random Forest  0.836066  0.864865
2                  SVM  0.836066  0.861111


In [ ]:
model_lr = LogisticRegression(max_iter=1000)
model_rf = RandomForestClassifier(n_estimators=400, random_state=42)
model_knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=11, weights='distance'))

model_lr.fit(X_train, y_train)
model_rf.fit(X_train, y_train)
model_knn.fit(X_train, y_train)

pred_lr = model_lr.predict(X_test)
pred_rf = model_rf.predict(X_test)
pred_knn = model_knn.predict(X_test)

acc_lr = accuracy_score(y_test, pred_lr)
f1_lr = f1_score(y_test, pred_lr)

acc_rf = accuracy_score(y_test, pred_rf)
f1_rf = f1_score(y_test, pred_rf)

acc_knn = accuracy_score(y_test, pred_knn)
f1_knn = f1_score(y_test, pred_knn)

base_learners = [
    ('xgb', XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=5,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=42)),
    
    ('knn', make_pipeline(StandardScaler(),
                          KNeighborsClassifier(n_neighbors=11, weights='distance'))),
    
    ('rf', RandomForestClassifier(
        n_estimators=400, max_depth=None, min_samples_leaf=2,
        class_weight='balanced', random_state=42))
]
meta_model = LogisticRegression(C=1.0, penalty='l2', max_iter=200, solver='lbfgs')
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

stack_model = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_model,
    cv=skf,
    stack_method='predict_proba',
    passthrough=False,
    n_jobs=-1
)
stack_model.fit(X_train, y_train)
y_pred_stack = stack_model.predict(X_test)
acc_stack = accuracy_score(y_test, y_pred_stack)
f1_stack = f1_score(y_test, y_pred_stack)

results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'KNN', 'Stacked Model'],
    'Accuracy': [acc_lr, acc_rf, acc_knn, acc_stack],
    'F1 Score': [f1_lr, f1_rf, f1_knn, f1_stack]
})

print("\n Final Model Comparison:\n")
print(results.to_string(index=False))



 Final Model Comparison:

              Model  Accuracy  F1 Score
Logistic Regression  0.803279  0.833333
      Random Forest  0.819672  0.853333
                KNN  0.819672  0.849315
      Stacked Model  0.836066  0.861111
